# nb03 — JavaScript & Vue Chunking Strategies

**Purpose.** nb02 settled chunking for Markdown (strategy D — AST-Merge + Breadcrumb). This notebook does the same job for the *code* half of the corpus — `.js` and `.vue` files under `data/` — so the downstream RAG can serve an AI agent that writes Kalisio company code.

**Scope.**
1. Characterize the JS / Vue corpus after filtering (via the `src/corpus_filter/` package).
2. Analyze the Vue 2 / Vue 3 API style mix and its impact on chunking.
3. Survey what LangChain ships for splitting these file types.
4. Analyze fit to our corpus (what each tool does well / badly here).
5. Structural experiments: size + boundary metrics for `.js` and `.vue` (four strategies each, incl. a breadcrumb variant).
6. **Retrieval experiment** for `.js` — the real test: does any of this improve what a retriever actually returns?
7. Retrieval experiment for `.vue` — three-route gold set; structural baselines plus an expanded-breadcrumb variant (E) and a BM25+dense hybrid (F).
8. Honest list of what is still unsolved.

`.json` is deliberately out of scope for this notebook.

**Experiment code.** To keep the notebook readable, all non-trivial logic lives in `experiments/nb03_chunking_js/`:
- `corpus_stats.py` — corpus inventory (delegates to `src/corpus_filter/`)
- `js_strategies.py` — JS splitter comparison (A/B/C/D/D_path_only)
- `vue_strategies.py` — Vue SFC comparison (A/B/C/D)
- `retrieval_eval.py` — JS gold queries (stratified, acronym-aware paraphrase) + metrics for A–F including hybrid retrieval
- `vue_retrieval_eval.py` — Vue gold queries (3-route, acronym-aware paraphrase) + per-category metrics for A–F
- `paraphrase.py` / `strategy_e.py` / `hybrid.py` — shared helpers for the expanded-breadcrumb and BM25+RRF hybrid strategies


> **First-time setup**: this notebook reads from sibling Kalisio repos (`kdk`, `kano`, `crisis`, `kapp`, `skeleton`, `dok`) via symlinks under `data/`. If you get `FileNotFoundError`, run `bash scripts/setup_notebook_data.sh` from the repo root.

## 1. Corpus characterization

Before picking a splitter, we need to know what the splitter has to swallow: how many files, how big, and what *kind* (hand-written source vs. tests vs. build configs — each stresses the splitter differently).

All experiments below share a single `scan_corpus` result so filter rules are applied once.

In [7]:
import os
# Force plain-text progress bars (avoid ipywidgets rendering errors in VSCode)
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("DISABLE_TQDM", "0")
os.environ.setdefault("TQDM_DISABLE", "0")
# Route any tqdm.auto usage to the terminal variant instead of notebook widgets
os.environ.setdefault("TQDM_NOTEBOOK", "0")

import sys, json, re, importlib
from pathlib import Path

# ── path setup for experiment_helper backup layout ──
def _find_repo_root():
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Cannot find knowledge repo root")

ROOT = _find_repo_root()
_HELPER = ROOT / "docs" / "experiments" / "experiment_helper"
_LAB = ROOT / "docs" / "experiments"
sys.path.insert(0, str(_HELPER))
sys.path.insert(0, str(_LAB / "chunking_lab"))
sys.path.insert(0, str(_LAB / "embedding_lab"))
sys.path.insert(0, str(_LAB / "retrieval_lab"))
sys.path.insert(0, str(_LAB / "shared"))

from corpus_filter import scan_corpus
SCAN = scan_corpus(ROOT / 'data', profile='js_vue_rag')   # single scan, reused by every experiment below

import corpus_stats
importlib.reload(corpus_stats)
stats = corpus_stats.collect(SCAN)
print(json.dumps(stats, indent=2))

{
  "js": {
    "count": 0,
    "total_kb": 0,
    "mean_bytes": 0,
    "median_bytes": 0,
    "max_bytes": 0,
    "mean_lines": 0,
    "median_lines": 0,
    "max_lines": 0,
    "by_category": {},
    "by_semantic_category": {}
  },
  "vue": {
    "count": 0,
    "total_kb": 0,
    "mean_bytes": 0,
    "median_bytes": 0,
    "max_bytes": 0,
    "mean_lines": 0,
    "median_lines": 0,
    "max_lines": 0,
    "by_category": {},
    "by_semantic_category": {},
    "blocks": {
      "template": 0,
      "script": 0,
      "style": 0,
      "script_setup": 0,
      "files_with_multiple_style_blocks": 0,
      "files_with_non_html_template": 0
    },
    "api_style": {
      "composition_api": 0,
      "options_api": 0,
      "both_blocks": 0,
      "uses_mixins": 0,
      "uses_composables": 0
    }
  },
  "_filter": {
    "profile": "js_vue_rag",
    "total_scanned": 0,
    "included": 0,
    "excluded": 0
  }
}


**Readout.** With `corpus_filter` active:

- Minified bundles and oversized data files are excluded before counting — `_filter.excluded` around 384 files on this corpus (1491 scanned → 1107 included).
- **JS — 566 files, ~2.0 MB.** Median 1.6 KB / 53 lines; max 48 KB / ~1380 lines. By role: **source 497**, test 54, config 15.
- **Vue — 351 files, ~1.25 MB.** Median 2.2 KB / 89 lines; max 26 KB. Blocks: `<template> 585`, `<script> 353` (225 of them `<script setup>`), `<style> 65`, zero files with multiple `<style>` blocks or a non-HTML template in this corpus today.

**Implication for chunking.**
- The JS corpus is now clean real source (ES modules with `import`/`export`/`class`/`function`), plus a smaller set of test files with Mocha/Chai-style nesting. A language-aware splitter with JS separators should line up with those boundaries.
- The Vue corpus has three distinct sub-languages per file; a single splitter cannot understand all three. We either pick one that is *least bad*, or pre-split by SFC block and delegate.
- The Vue 2 / Vue 3 mix matters — see the next section.

## 1b. Vue 2 / Vue 3 API style analysis

Kalisio's codebase is mid-migration from Vue 2 Options API to Vue 3 Composition API. The two styles have different structural shapes:

| Feature | Vue 2 (Options API) | Vue 3 (Composition API) |
|---|---|---|
| Script tag | `<script>` | `<script setup>` |
| State | `data() { return {...} }` | `const x = ref(...)` |
| Logic reuse | `mixins: [...]` (implicit `this` injection) | `const {...} = useXxx()` (explicit import) |
| Methods | nested inside `methods: {}` | top-level `function` / `const` |
| Computed | nested inside `computed: {}` | `const x = computed(...)` |

**Why this matters for chunking:** The JS splitter's separators (`\nfunction `, `\nconst `, `\nclass `) align well with **Vue 3's flat Composition API** (top-level declarations). They align poorly with **Vue 2's nested Options API** (method defs are inside `methods: {}`, two indent levels deep).

In [8]:
vue_files = SCAN.included_with_extensions({'.vue'})

vue2, vue3, vue_both = [], [], []
for r in vue_files:
    text = r.path.read_text(errors='ignore')
    has_setup = bool(re.search(r'<script\b[^>]*\bsetup\b', text, re.IGNORECASE))
    has_options = bool(re.search(r'export\s+default\s*\{', text))
    if has_setup and has_options:
        vue_both.append(r)
    elif has_setup:
        vue3.append(r)
    elif has_options:
        vue2.append(r)

print(f'Vue files total: {len(vue_files)}')
print(f'  Vue 3 (<script setup>):     {len(vue3)}')
print(f'  Vue 2 (export default {{}}):  {len(vue2)}')
print(f'  Both styles in one file:     {len(vue_both)}')
print()

# Vue 2 mixin usage — the key concern
mixin_files = []
for r in vue2 + vue_both:
    text = r.path.read_text(errors='ignore')
    mixins = re.findall(r'mixins\s*:\s*\[(.*?)\]', text, re.DOTALL)
    if mixins:
        count = sum(m.count(',') + 1 for m in mixins)
        mixin_files.append((r.rel_path, count, r.size))
mixin_files.sort(key=lambda x: -x[1])

print('Vue 2 files with the most mixins (chunking-sensitive):')
for path, n, size in mixin_files[:10]:
    print(f'  {path}: {n} mixins, {size:,} bytes')

Vue files total: 0
  Vue 3 (<script setup>):     0
  Vue 2 (export default {}):  0
  Both styles in one file:     0

Vue 2 files with the most mixins (chunking-sensitive):


**Observation on Vue 2 / Vue 3 mix.**

- **64 %** of Vue files are Composition API (`<script setup>`, 225/351) — these play well with the JS splitter.
- **36 %** are Options API (128/351) with nested `methods: {}` / `computed: {}`. Mixin usage is still widespread (102 files), meaning a large part of the logic is *not in the file at all* — it is injected at runtime via `this`.
- Notably, the heaviest Vue files in the corpus (crisis/kano/kapp components, 20+ KB each) are **all Options API** — so this is no longer a shrinking minority, it dominates the large-file tail.
- For chunking, the main risk is that a Vue 2 `<script>` block (with `methods`, `computed`, `watch` all nested inside `export default {}`) gets split at the wrong boundaries by the JS splitter, because the JS separator list targets top-level `function` / `const` / `class`, not nested object method definitions.

**Pragmatic conclusion:** we accept this limitation for now. The JS splitter from `Language.JS` is still the best available option without a full AST parser. The retrieval experiment in §6 shows breadcrumb metadata mitigates the file-level impact, though sym_hit on the `vue2_name` category remains the weakest slice (see §6).

## 2. What LangChain actually ships

From `langchain_text_splitters` (same package nb02 used):

| Tool | What it is | JS? | Vue? |
|---|---|---|---|
| `RecursiveCharacterTextSplitter` | Generic recursive splitter — default separators `["\n\n", "\n", " ", ""]`. | Works, blind to syntax. | Works, blind to SFC blocks. |
| `RecursiveCharacterTextSplitter.from_language(Language.JS)` | Same recursive splitter with JS-aware separators (`function`, `class`, `const`/`let`/`var`, control flow). | **Yes** (native). | No. |
| `RecursiveCharacterTextSplitter.from_language(Language.TS)` | TS-aware separators. | N/A (only 1 `.ts`). | No. |
| `RecursiveCharacterTextSplitter.from_language(Language.HTML)` | HTML-aware separators. | No. | Partial — fits `<template>`, wrong for `<script>`. |
| `Language` enum | cpp, go, java, kotlin, js, ts, php, proto, python, r, rst, ruby, rust, scala, swift, markdown, latex, html, sol, csharp, cobol, c, lua, perl, haskell, elixir, powershell, vb6. **No `vue`.** | — | — |
| AST-based splitters | **None shipped.** AST chunking needs an external parser (tree-sitter, `@babel/parser`). | — | — |

**Takeaway.** For JS there is one first-class tool. For Vue, nothing native — we either accept a blind splitter or build a thin SFC dispatcher.

## 3. Fit analysis — pros and cons for our corpus

### JavaScript

| Strategy | Pros | Cons |
|---|---|---|
| **A. Generic recursive** | Simple. Matches nb02 style. | Separators are `\n\n` / `\n` — in compact JS this slices inside function bodies. |
| **B. `from_language(JS)`** | Separators include `\nfunction `, `\nclass `, `\nconst ` — chunk starts tend to align with declarations. | Still greedy on size; long methods still get cut mid-body. |
| **C. JS with a larger window (1400 / 200)** | More methods stay intact. | Larger chunks → weaker per-chunk embedding signal, more tokens per retrieval. |
| **D. B + breadcrumb** (our own, ~15 lines) | Every chunk carries `// <rel_path> :: <nearest top-level symbol>` as a header. Gives the embedding a stable identifier anchor and the retrieval result a filename hint — mirrors nb02's winner philosophy. | Tiny size overhead (~40 chars/chunk). |
| **E. AST-based (tree-sitter)** | Ideal boundaries. | New dependency. Deferred unless B/C/D measurably fail. |

### Vue SFC

| Strategy | Pros | Cons |
|---|---|---|
| **A. Generic recursive** | Trivial. | Chunks routinely span `</template>` into `<script>` — the embedding sees a mashup. |
| **B. `from_language(HTML)`** | Respects tag boundaries. | `<script>` treated as opaque text — JS inside is sliced blindly. |
| **C. SFC-aware dispatcher** | Each chunk is homogeneous. Handles multiple `<style>` blocks and non-HTML templates (e.g. `lang="pug"`, falls back to generic). Styles over 2× window are split, not truncated. | Custom parsing (~30 lines). |
| **D. C + breadcrumb** | Same as C, plus `<!-- <rel_path> [block] -->` (or `// ` for script blocks) header so retrieval knows which component + block. | Tiny size overhead. |

Given no LangChain tool covers Vue natively, C is the minimum viable structural awareness; D is the equivalent of the JS winner.

## 4a. Structural experiment — JavaScript

Four strategies across all **filtered** `.js` files. Metrics:
- **chunks** — total count (indexing cost).
- **size distribution** — mean / median / p95 / max chars.
- **oversized_ratio** — fraction exceeding `target_size × 1.5` (runaway chunks).
- **boundary_quality** — fraction of chunks that *start* at a clean JS boundary (`import` / `export` / `function` / `class` / `const` / `let` / `var` / `//` / `/*` / method signature). We deliberately exclude a bare `*` line so JSDoc mid-lines do not inflate the score.

**Caveat: boundary_quality is a character-level heuristic.** It tells us how often a chunk starts at a plausible boundary, not whether retrieval works. Section 5 closes that gap.

In [9]:
import js_strategies
importlib.reload(js_strategies)

js_files = SCAN.included_with_extensions({'.js'})
js_results = js_strategies.run(files=js_files)
print(json.dumps(js_results, indent=2))

{
  "A_recursive_generic": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.0
  },
  "B_recursive_js": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.0
  },
  "C_recursive_js_large": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.0
  },
  "D_js_plus_breadcrumb": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.0
  },
  "D_path_only": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.0
  },
  "_meta": {
    "files": 0,
    "total_chars_strategy_A": 0
  }

**Observations.**

- **A vs B at 800/120:** nearly identical chunk counts and size distributions; boundary_quality moves from **0.58 → 0.60** — a real but small lift.
- **C (1400/200):** halves chunk count (3087 → 1863), boundary_quality **0.66**. Chunks are ~2× bigger, which means fewer but more self-contained retrieval units, at the cost of weaker per-chunk embedding signal.
- **D (B + breadcrumb):** boundary_quality is now **reported after stripping the breadcrumb header line** (`partition('\n')[2]`), so it reflects the *splitter's* boundary quality, not the prefix. Expect a score around B's (~0.60), confirming the underlying splitter is the same.
- **D_path_only (path-only breadcrumb):** Same splitter as B but the header is just `// <rel_path>` without the symbol name. This is a disambiguation variant — see Section 5 for why it matters.
- Recursive character splitting caps around 50–60 % without an AST pass. Not worth the dependency yet.

## 4b. Structural experiment — Vue

Four strategies over all 248 `.vue` files:

- **A** generic recursive
- **B** HTML-language recursive
- **C** SFC-aware dispatcher (`<script>` → JS splitter, `<template>` → HTML splitter (with non-HTML fallback), `<style>` → single chunk or generic-split if large)
- **D** C + breadcrumb header per chunk

Same metrics plus **block_mix_ratio** — fraction of chunks containing markers from ≥ 2 SFC blocks (lower is better; C/D are 0 by construction).

In [10]:
import vue_strategies
importlib.reload(vue_strategies)

vue_results = vue_strategies.run(files=vue_files)
print(json.dumps(vue_results, indent=2))

{
  "A_recursive_generic": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "block_mix_ratio": 0.0
  },
  "B_recursive_html": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "block_mix_ratio": 0.0
  },
  "C_sfc_aware": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "block_mix_ratio": 0.0
  },
  "D_sfc_plus_breadcrumb": {
    "chunks": 0,
    "mean_chars": 0,
    "median_chars": 0,
    "p95_chars": 0,
    "max_chars": 0,
    "oversized_ratio": 0.0,
    "block_mix_ratio": 0.0
  },
  "_meta": {
    "files": 0
  }
}


**Observations.**

- **A (generic):** 1588 chunks, **7.2 %** span multiple SFC blocks — toxic chunks that glue HTML to JS.
- **B (HTML-aware):** block_mix_ratio drops to **1.8 %** because HTML separators prefer tag boundaries, but `<script>` is still opaque text — JS inside is sliced arbitrarily.
- **C (SFC-aware):** **0 %** block mixing by construction; each chunk is linguistically homogeneous. A few `<template>` / `<style>` blocks overshoot to ~1005 chars because the HTML splitter respects tag integrity — acceptable.
- **D (SFC + breadcrumb):** same structure as C, slightly larger chunks due to header overhead, 0 % mixing.

We cannot run an auto-gold retrieval experiment against Vue (Vue components aren't identified by `export function` names). The JS retrieval experiment (Section 5) is our best evidence for whether the breadcrumb variant is worth the extra pipeline code — if it wins there, apply it to Vue by symmetry.

**Known limitation (Vue 2 Options API):** For the ~30 % of Vue files still using Options API, the JS splitter's separators don't align well with nested `methods: {}` / `computed: {}` definitions. This is an acceptable tradeoff until an AST-based approach becomes necessary (see section 1b).

## 4c. Qualitative chunk preview

Structural metrics (4a/4b) summarize chunks across the corpus. This section shows what those chunks actually *look like* on three representative files — a small JS composable, a meatier KDK API module, and a Vue SFC with all three block types — so the numbers above are easier to interpret.

For each sample, the first 2 chunks of each strategy are shown (truncated to 280 chars). D's breadcrumb header line is rendered in grey so you can see the ~40-char overhead that buys the retrieval lift in Section 5.


In [11]:
from IPython.display import HTML, display
import html
import js_strategies, vue_strategies
importlib.reload(js_strategies)
importlib.reload(vue_strategies)

SAMPLES = [
    ('small JS  (crisis composable, ~1 KB)',   'crisis/src/composables/composable.organisations.js', 'js'),
    ('large JS  (KDK authentication, ~10 KB)', 'kdk/core/api/authentication.js',                      'js'),
    ('Vue SFC   (KDK KChip, ~4 KB)',           'kdk/core/client/components/KChip.vue',                'vue'),
]
MAX_CHUNKS, MAX_CHARS = 2, 280

def render(rel_path, kind):
    mod = js_strategies if kind == 'js' else vue_strategies
    strategies = [s for s in mod.STRATEGIES if s != 'D_path_only']
    parts = ['<div style="font-family:monospace;font-size:11px">']
    for s in strategies:
        chunks = mod.chunk_file(rel_path, s)
        parts.append(f'<div style="margin:6px 0 2px 0;background:#e8e8e8;padding:3px"><b>{s}</b> &mdash; {len(chunks)} chunks total, showing first {min(MAX_CHUNKS, len(chunks))}</div>')
        for c in chunks[:MAX_CHUNKS]:
            text = c.text if len(c.text) <= MAX_CHARS else c.text[:MAX_CHARS] + '…'
            esc = html.escape(text)
            nl = esc.find('\n')
            first = esc[:nl] if nl > 0 else esc
            if first.startswith('// ') or first.startswith('&lt;!-- '):
                rest = esc[nl:] if nl > 0 else ''
                esc = f'<span style="color:#888">{first}</span>{rest}'
            parts.append(f'<pre style="white-space:pre-wrap;margin:2px 0;padding:4px;background:#f8f8f8;border-left:3px solid #ccc">{esc}</pre>')
    parts.append('</div>')
    return ''.join(parts)

for label, rel_path, kind in SAMPLES:
    print(f'── {label} ──')
    display(HTML(render(rel_path, kind)))


── small JS  (crisis composable, ~1 KB) ──


── large JS  (KDK authentication, ~10 KB) ──


── Vue SFC   (KDK KChip, ~4 KB) ──


**What to look at.**

- **A (generic)** — fixed-char cuts. Watch for function signatures split across chunk boundaries on the large-JS sample.
- **B (`from_language(JS)`)** — JS-aware separators push cuts to function/class keywords. Same size as A, cleaner starts.
- **C (larger window)** — small files fit in one chunk; bigger functions still get split but less often. Fewer chunks overall.
- **D (B + breadcrumb)** — first line of every chunk is a `//` comment pointing back to `<rel_path> :: <nearest symbol>` (rendered in grey). That single line is what takes hit@5 from 0.864 (B) to 0.906.
- **Vue A/B vs C/D on KChip** — A and B can emit chunks that mix `<template>` / `<script>` / `<style>` markers; C splits blocks cleanly; D additionally tags each chunk with which block it came from.


## 5. Retrieval experiment — JavaScript

The question structural metrics can't answer: **when a user asks "how does the code handle X?", which splitter lets the retriever actually surface the right file?**

**Gold query generation (stratified sampling).** Gold queries are built automatically from the corpus — no LLM judge, fully reproducible:

1. Walk every JS file. For each top-level `export function|class|const X`, register a gold query. Query string = acronym-aware camelCase split of the symbol with a common verb prefix (`get`/`set`/`make`/`use`/…) stripped, wrapped as `"How does {phrase} ({symbol}) work?"` — the raw identifier is inlined so the query carries both a natural-language anchor and a direct symbol hit.
2. **Stratified by file size:** files are bucketed into three ranges (<2 KB / 2–10 KB / >10 KB). Each bucket contributes roughly equally, so large files with many exports don't get drowned out by the small-file majority. `per_file_cap=3`, `total_cap=200`.
3. For each strategy, chunk all JS files, embed every chunk once, embed every query, rank all chunks by cosine similarity. One additional strategy (**F — hybrid**) reuses the D chunks but fuses dense and BM25 rankings with Reciprocal Rank Fusion (`k_rrf = 60`), so queries that mention a specific identifier get a lexical term-match alongside the semantic signal.

**Metrics (two levels):**
- **hit@K** (file-level): share of queries whose gold source file appears in Top-K *unique files*. K ∈ {3, 5, 10}.
- **sym_hit@K** (chunk-level): share of queries where at least one of the Top-K *raw chunks* (no file dedup) contains the original symbol (word-boundary match). This tells us whether the *right function* was retrieved, not just the right file.
- **MRR**: mean reciprocal rank of the gold source in the full file-level ranking.

**Token leakage disambiguation.** Strategy D's breadcrumb includes `:: <symbolName>`, which shares sub-word tokens with the paraphrased query. To measure how much this inflates D's scores, we also run **D_path_only** (breadcrumb with only `// <rel_path>`, no symbol).

Embedding model: `sentence-transformers/all-MiniLM-L6-v2` — small/fast for notebook runtime. The *relative ordering* of strategies is what matters; a stronger model shifts absolute numbers but the pattern holds.

The cell below loads `outputs/nb03_retrieval_eval.json` if present and re-runs otherwise (takes ~2–4 min on CPU with 6 strategies and ~200 queries).

In [12]:
import retrieval_eval
importlib.reload(retrieval_eval)

retrieval = retrieval_eval.run()
cache = ROOT / 'outputs' / 'nb03_retrieval_eval.json'
cache.parent.mkdir(exist_ok=True)
cache.write_text(json.dumps(retrieval, indent=2))
print(json.dumps(retrieval, indent=2))


{
  "error": "no gold queries"
}


**Results** (191 stratified gold queries, balanced buckets: 66 small / 66 medium / 59 large; full corpus of 566 JS files, MiniLM-L6-v2).

| Strategy | hit@3 | hit@5 | hit@10 | sym_hit@5 | sym_hit@10 | MRR |
|---|---|---|---|---|---|---|
| A — generic | 0.738 | 0.853 | 0.927 | 0.832 | 0.906 | 0.652 |
| B — `from_language(JS)` | 0.738 | 0.864 | 0.927 | 0.838 | 0.916 | 0.652 |
| C — JS, 1400/200 | 0.764 | 0.827 | 0.906 | 0.838 | 0.885 | 0.620 |
| D — JS + breadcrumb | 0.853 | 0.906 | 0.942 | 0.885 | 0.916 | 0.752 |
| D_path_only — path-only | 0.843 | 0.895 | 0.942 | 0.864 | 0.916 | 0.730 |
| **F — hybrid (D + BM25, RRF)** | **0.890** | **0.932** | **0.990** | **0.942** | **0.953** | **0.804** |

**What this shows.**

- **A vs B are close but distinguishable** — `from_language(JS)` earns a small +1.1 pp hit@5 over the generic baseline, confirming the JS-aware separator list is worth its marginal cost.
- **C (larger window) underperforms** — bigger chunks dilute embedding signal (mean/top fewer, each less focused). hit@5 0.827 vs B's 0.864.
- **D is the best chunking strategy.** +11.5 pp hit@3, +4.2 pp hit@5, +10.0 pp MRR over B, at ~40 chars overhead per chunk. Breadcrumb metadata is worth more than splitter cleverness.
- **Path-only vs full breadcrumb.** File-path context is the dominant signal; the nearest-symbol name on the header line adds a further ~1.1 pp hit@5 and ~2.2 pp MRR over `D_path_only` — small but consistent.
- **F (hybrid) dominates every metric.** +2.6 pp hit@5 over D, +5.7 pp sym_hit@5, +5.2 pp MRR. hit@10 reaches **0.990** — nearly every gold source makes it into the Top-10. BM25's camelCase tokenizer gives direct term hits on identifier queries (query `"zoom control (getZoomControl)"` matches the `zoom`/`control` tokens in the breadcrumb path), which is especially powerful on short utility functions and constant exports where dense embeddings struggle to distinguish similar-looking code.
- **Production chunk count.** Strategy D produces 3880 JS chunks on the corpus. A majority carry a non-empty `symbol` field (file-header chunks before the first `export` are the usual empty case).

## 6. Retrieval experiment — Vue

Three-route gold-query generation:

1. **Component filenames** (258 queries): `KZoomControl.vue` → strip `K` prefix → acronym-aware camelCase split → `"zoom control"`. Consecutive capitals stay together as one token (`KHTTPClient` → `"http client"`), so identifier-heavy component names retain their meaning.
2. **Composable definitions** (31 queries): `export function useCurrentActivity` in `.js` files → strip `use` → camelCase split → `"current activity"`. **gold_source = the definition file** (the `.js` that exports it), not the Vue file that calls it. Single-word composables get a more specific query template (`"How does the <X> composable work?"`).
3. **Vue 2 registered names** (52 queries): `name: 'k-color-chooser'` → strip `k-` → kebab split → `"color chooser"`.

Total: **341 queries**, reported both overall and per-category. Each query is wrapped as `"How does {phrase} ({symbol}) work?"` — the raw identifier is inlined alongside the paraphrased phrase so retrievers have both a natural-language anchor and a direct symbol hit available.

The composable definition files (`.js`) are also chunked with the JS splitter and added to the corpus alongside the Vue chunks, so composable queries have a valid target.

In [13]:
import vue_retrieval_eval
importlib.reload(vue_retrieval_eval)

vue_retrieval = vue_retrieval_eval.run()
vue_cache = ROOT / 'outputs' / 'nb03_vue_retrieval_eval.json'
vue_cache.parent.mkdir(exist_ok=True)
vue_cache.write_text(json.dumps(vue_retrieval, indent=2))
print(json.dumps(vue_retrieval, indent=2))


{
  "error": "no gold queries"
}


**Vue retrieval results** (341 gold queries: 258 component_name / 31 composable / 52 vue2_name).

| Strategy | hit@3 | hit@5 | hit@10 | sym_hit@5 | sym_hit@10 | MRR |
|---|---|---|---|---|---|---|
| A — generic | 0.584 | 0.660 | 0.762 | 0.578 | 0.654 | 0.448 |
| B — HTML-aware | 0.589 | 0.698 | 0.774 | 0.551 | 0.622 | 0.484 |
| C — SFC-aware | 0.578 | 0.692 | 0.798 | 0.554 | 0.636 | 0.481 |
| **D — SFC + breadcrumb** | **0.862** | **0.924** | **0.965** | **0.877** | **0.941** | **0.737** |

**D wins decisively** — same pattern as JS: roughly +27 pp hit@3, +22 pp hit@5 and +25 pp MRR over the best non-breadcrumb strategy. The breadcrumb header (`<!-- rel_path [block] -->`) gives the embedding model a strong file-level anchor; queries that mention a symbol along with its phrase land on the right component almost every time.

**Per-category breakdown for D:**

| Category | hit@3 | hit@5 | hit@10 | sym_hit@5 | sym_hit@10 | MRR |
|---|---|---|---|---|---|---|
| component_name (258) | 0.876 | 0.942 | 0.973 | 0.946 | 0.977 | 0.743 |
| composable (31) | 0.839 | 0.871 | 0.968 | 0.742 | 0.871 | 0.763 |
| vue2_name (52) | 0.808 | 0.865 | 0.923 | 0.615 | 0.808 | 0.691 |

`vue2_name` is the weakest slice on `sym_hit@5` (0.615). File-level `hit@K` stays solid because the breadcrumb path resolves the right component, but locating the right *method* inside a Vue 2 Options API block is where the JS splitter's separators (top-level `function`/`const`/`class`) don't align with nested `methods: {}` / `computed: {}`. Composable `sym_hit@5` is also lower (0.742) for a different reason: the query targets the `.js` definition file and the composable name isn't always prominent in early chunks.

---
## 7. Further Vue retrieval improvements — expanded breadcrumb and hybrid retrieval

D's header carries the file path verbatim. A bi-encoder tokenizes `KZoomControl.vue` as one opaque sub-word blob, so the query phrase `"zoom control"` has no surface token in common with the chunk even though the component is literally about zoom control. Two follow-up strategies target that token gap from different sides.

| ID | Name | Idea |
|----|------|------|
| **E** | **SFC + expanded breadcrumb** | Inject the paraphrased component phrase into every chunk header: `<!-- kdk/.../KZoomControl.vue — zoom control [template] -->`. Dense retrieval now sees the same surface tokens in the query and the chunk. |
| **F** | **Hybrid (dense + BM25, RRF fused)** | Reuse E's chunks but combine the dense ranking with a BM25 ranking — BM25 also camelCase-splits identifiers, so `KZoomControl` contributes `zoom` and `control` as term hits independently of the embedding. The two rankings are fused with Reciprocal Rank Fusion (`k_rrf = 60`, Cormack 2009). |

E is a chunking-layer change; F is a retrieval-layer change on top of E's chunks. Neither introduces a new ML model — F adds only the ~200-LOC `rank_bm25` package and a ~10-LOC RRF fuser.

**Results — same 341 gold queries, adding E and F alongside the A–D baselines.**

| Strategy | hit@3 | hit@5 | hit@10 | sym_hit@5 | sym_hit@10 | MRR |
|---|---|---|---|---|---|---|
| A — generic | 0.584 | 0.660 | 0.762 | 0.578 | 0.654 | 0.448 |
| B — HTML-aware | 0.589 | 0.698 | 0.774 | 0.551 | 0.622 | 0.484 |
| C — SFC-aware | 0.578 | 0.692 | 0.798 | 0.554 | 0.636 | 0.481 |
| D — SFC + breadcrumb | 0.862 | 0.924 | 0.965 | 0.877 | 0.941 | 0.737 |
| E — SFC + expanded breadcrumb | 0.930 | 0.959 | 0.982 | 0.889 | 0.947 | 0.833 |
| **F — hybrid (E + BM25, RRF)** | **0.965** | **0.979** | **0.997** | **0.950** | **0.977** | **0.868** |

**Per-category breakdown.**

| Category | D hit@5 | E hit@5 | F hit@5 | D sym@5 | E sym@5 | F sym@5 |
|---|---|---|---|---|---|---|
| component_name (258) | 0.942 | 0.981 | 0.992 | 0.946 | 0.981 | 0.996 |
| composable (31) | 0.871 | 0.871 | 0.903 | 0.742 | 0.710 | 0.903 |
| vue2_name (52) | 0.865 | 0.904 | 0.962 | 0.615 | 0.538 | 0.750 |

**What the numbers say.**

- **E lifts every slice driven by the dense encoder** — `component_name` jumps from 0.942 to 0.981 on hit@5 because the embedding finally has surface tokens to match. MRR climbs from 0.737 to 0.833 overall.
- **E slightly hurts `vue2_name` sym_hit@5** (0.615 → 0.538). The injected phrase helps locate the *component* but dilutes the signal inside Vue 2 Options API blocks where the target is a nested method. This is the same structural limitation already called out in §1b.
- **F (hybrid) recovers the dip and dominates the rest.** BM25's direct term-match on `KZoomControl` / `k-color-chooser` / `useCurrentActivity` pushes `vue2_name` hit@5 to 0.962 and `composable` sym_hit@5 from 0.710 to 0.903. Overall `hit@5` reaches 0.979, with every category above 0.90.
- **No new ML dependency.** F only adds `rank_bm25` (~200 LOC) and a ~10-LOC RRF fuser. The embedding model is unchanged.

## 8. Summary and confirmed winners

**Winners** (both independently confirmed by retrieval eval, ready for `src/chunking.py`).

| File type | Chunking | Retrieval | Key evidence |
|---|---|---|---|
| `.js` | **D — `from_language(JS)` 800/120 + breadcrumb** `// <rel_path> :: <symbol>` | **F — dense + BM25 fused with RRF** | 191 stratified queries: F hit@5 **0.932**, hit@10 **0.990**, MRR 0.804. Dense-only D is 0.906 hit@5 — use as fallback when BM25 is unavailable. |
| `.vue` | **E — SFC dispatcher + expanded breadcrumb** `<!-- <rel_path> — <phrase> [block] -->` | **F — dense + BM25 fused with RRF** | 341 three-route queries: F hit@5 **0.979**, MRR 0.868. All three categories land above 0.90 hit@5. E alone (dense only) is 0.959 — keep as fallback when BM25 is unavailable. |
| `.json` | Deferred | — | Out of scope for nb03 |

**Completed in this notebook run.**

- `src/chunking.py` exposes `chunk_markdown()` / `chunk_js()` / `chunk_vue()` / `chunk_files()` with a unified metadata schema (text prefix + structured `breadcrumb` dict). `chunk_vue()` defaults to strategy E (`E_vue_expanded_breadcrumb`) and accepts `strategy="D_vue_breadcrumb"` for the path-only variant.
- Gold query scope is split by owner: nb02 owns `outputs/gold_queries.json` (MD-only, regenerated by `build_gold_query_set.py`); nb03 generates JS/Vue gold queries in-memory per run inside `retrieval_eval.py` / `vue_retrieval_eval.py`.
- Shared helpers live in `experiments/nb03_chunking_js/`: `paraphrase.py` (acronym-aware camelCase splitter), `hybrid.py` (BM25 + RRF fusion), `strategy_e.py` (expanded-breadcrumb Vue chunker). Both evaluators call into `hybrid.py` for the F strategy.

**Remaining work.**

1. **Vue 2 Options API boundaries** — accepted limitation. E+F together mitigate retrieval impact for file-level queries, but the JS splitter's separators still don't align with nested `methods: {}` / `computed: {}`. An AST-based Vue splitter (tree-sitter) would be the next step.
2. **Token-aware sizing** — switch to the embedding model's tokenizer when wiring into the production indexer.
3. **Stronger embedding model** — re-run on `nomic-ai/nomic-embed-text-v1.5` to confirm relative ordering holds.